# Notebook 11: Ensemble Methods — Random Forest and Boosting
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Understand bagging and boosting as complementary strategies
- Explain why Random Forest reduces variance
- Implement AdaBoost and Gradient Boosting
- Compare ensemble methods on real data
- Use feature importances from Random Forest

## 1. Why Ensembles?

If you have 25 independent classifiers each with 70% accuracy and take a majority vote:

$$P(\text{majority correct}) = \sum_{k=13}^{25} \binom{25}{k} 0.7^k 0.3^{25-k} \approx 0.997$$

The key ingredient is **diversity** — the classifiers must make different mistakes.

### Two Strategies
| | **Bagging** | **Boosting** |
|--|------------|-------------|
| Training | Parallel | Sequential |
| Focus | All samples equally | Misclassified samples more |
| Reduces | Variance | Bias |
| Example | Random Forest | AdaBoost, GradientBoosting |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, BaggingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Random Forest = Bagging + Random Feature Selection

**Bagging**: sample B bootstrap datasets → train one tree on each → vote.

**Random Forest** adds: at each split, only consider $\sqrt{p}$ randomly chosen features. This **decorrelates** the trees — they make more independent errors, so the majority vote is stronger.

In [ ]:
models = [
    ('Single Tree',   DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Bagging (50)',  BaggingClassifier(DecisionTreeClassifier(max_depth=5),
                                        n_estimators=50, random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
]

print(f'{"Model":<25} Train    Test    CV-5')
print('-' * 52)
for name, m in models:
    m.fit(X_tr, y_tr)
    tr = m.score(X_tr, y_tr)
    te = m.score(X_te, y_te)
    cv = cross_val_score(m, X, y, cv=5).mean()
    print(f'{name:<25} {tr:.4f}   {te:.4f}   {cv:.4f}')

In [ ]:
# Effect of number of trees
n_range = [1, 5, 10, 25, 50, 100, 200]
accs = []
for n in n_range:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_tr, y_tr)
    accs.append(rf.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(n_range, accs, marker='o', linewidth=2)
ax.set_xlabel('Number of trees (log scale)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Random Forest: Accuracy vs Number of Trees')
plt.tight_layout(); plt.show()

In [ ]:
# Feature importances with uncertainty
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
imps = rf.feature_importances_
std  = np.std([t.feature_importances_ for t in rf.estimators_], axis=0)
idx  = np.argsort(imps)[::-1][:12]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(12), imps[idx], yerr=std[idx], capsize=3, color='steelblue')
ax.set_xticks(range(12))
ax.set_xticklabels([cancer.feature_names[i] for i in idx], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Importance ± std')
ax.set_title('Random Forest Feature Importances')
plt.tight_layout(); plt.show()

## 3. AdaBoost

**Algorithm**:
1. All samples start with equal weight
2. Train a weak classifier (often a depth-1 stump)
3. Increase weights of misclassified points
4. Repeat T rounds
5. Final prediction: weighted vote of all classifiers

The weight of classifier $m$ in the final vote is $\alpha_m = \frac{1}{2}\ln\frac{1-\epsilon_m}{\epsilon_m}$ where $\epsilon_m$ is its weighted error rate.

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=200, learning_rate=1.0, random_state=42
)
ada.fit(X_tr, y_tr)

staged_tr = list(ada.staged_score(X_tr, y_tr))
staged_te = list(ada.staged_score(X_te, y_te))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(staged_tr, label='Train')
ax.plot(staged_te, label='Test')
ax.set_xlabel('Boosting rounds')
ax.set_ylabel('Accuracy')
ax.set_title('AdaBoost: Staged Accuracy')
ax.legend()
plt.tight_layout(); plt.show()
print(f'AdaBoost final test accuracy: {ada.score(X_te, y_te):.4f}')

## 4. Gradient Boosting

Each new tree is fitted to the **negative gradient of the loss** — i.e., the residual errors of the current ensemble:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

**Key hyperparameters**:
- `n_estimators`: number of trees
- `learning_rate` $\eta$: shrinkage (lower = need more trees)
- `max_depth`: typically 3–5
- `subsample`: fraction of samples per tree (< 1.0 adds randomness = Stochastic GB)

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05,
    max_depth=3, subsample=0.8, random_state=42
)
gb.fit(X_tr, y_tr)
staged_te_gb = list(gb.staged_score(X_te, y_te))
staged_tr_gb = list(gb.staged_score(X_tr, y_tr))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(staged_tr_gb, label='Train')
ax.plot(staged_te_gb, label='Test')
ax.set_xlabel('Boosting rounds')
ax.set_ylabel('Accuracy')
ax.set_title('Gradient Boosting: Staged Accuracy')
ax.legend()
plt.tight_layout(); plt.show()
print(f'Gradient Boosting test accuracy: {gb.score(X_te, y_te):.4f}')

## 5. Head-to-Head Comparison

In [ ]:
all_models = [
    ('Decision Tree',       DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, random_state=42)),
    ('AdaBoost',            AdaBoostClassifier(n_estimators=100, random_state=42)),
    ('Gradient Boosting',   GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('Logistic Regression', Pipeline([('sc', StandardScaler()),
                                       ('lr', LogisticRegression(max_iter=1000))])),
]

results = {}
for name, model in all_models:
    cv = cross_val_score(model, X, y, cv=10)
    results[name] = cv
    print(f'{name:<25} CV: {cv.mean():.4f} ± {cv.std():.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(list(results.values()), labels=list(results.keys()), patch_artist=True)
ax.set_ylabel('10-fold CV Accuracy')
ax.set_title('Model Comparison — Breast Cancer')
ax.set_xticklabels(list(results.keys()), rotation=30, ha='right')
plt.tight_layout(); plt.show()

## Exercises

1. Tune `RandomForestClassifier` with `GridSearchCV` over `n_estimators`, `max_depth`, and `max_features`. What combination wins?
2. Enable `oob_score=True` on Random Forest — it estimates generalisation error from out-of-bag samples without needing a validation set. Compare to 5-fold CV.
3. Try `subsample=[0.5, 0.7, 1.0]` in Gradient Boosting. Does subsampling help regularise the model?
4. Look up `HistGradientBoostingClassifier` in sklearn — the fast version. Compare training time on a large dataset.

## Reflection Questions
- Bagging reduces variance but not bias. Boosting reduces bias. How do these connect to the bias-variance trade-off?
- Why does Random Forest randomly select features at each split — what would happen without this?
- If both Random Forest and Gradient Boosting achieve similar accuracy, which would you prefer to deploy and why?

---
**Next →** `12_text_classification.ipynb`